In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score

In [2]:
import numpy as np
import pandas as pd

test_files = [
    'archive/CMaps/test_FD001.txt',
    'archive/CMaps/test_FD002.txt',
    'archive/CMaps/test_FD003.txt',
    'archive/CMaps/test_FD004.txt'
]

rul_files = [
    'archive/CMaps/RUL_FD001.txt',
    'archive/CMaps/RUL_FD002.txt',
    'archive/CMaps/RUL_FD003.txt',
    'archive/CMaps/RUL_FD004.txt'
]

for idx, (test_file, rul_file) in enumerate(zip(test_files, rul_files), start=1):

    # =========================
    # LOAD TEST DATA
    # =========================
    raw_data = pd.read_csv(test_file, sep=' ', header=None)
    raw_data = raw_data.drop([26, 27], axis=1)

    raw_data.columns = [
        'ID','Cycle','OpSet1','OpSet2','OpSet3',
        'SensorMeasure1','SensorMeasure2','SensorMeasure3','SensorMeasure4','SensorMeasure5',
        'SensorMeasure6','SensorMeasure7','SensorMeasure8','SensorMeasure9','SensorMeasure10',
        'SensorMeasure11','SensorMeasure12','SensorMeasure13','SensorMeasure14','SensorMeasure15',
        'SensorMeasure16','SensorMeasure17','SensorMeasure18','SensorMeasure19','SensorMeasure20','SensorMeasure21'
    ]

    # =========================
    # LOAD RUL
    # =========================
    cycle_ran_after = pd.read_csv(rul_file, sep=' ', header=None)
    cycle_ran_after = cycle_ran_after.drop([1], axis=1)
    cycle_ran_after = np.array(cycle_ran_after)

    # =========================
    # GET LAST CYCLE PER ENGINE
    # =========================
    given_no_of_cycles = raw_data.groupby('ID')['Cycle'].max().values.reshape(-1, 1)

    # =========================
    # COMPUTE EOL
    # =========================
    EOL = given_no_of_cycles + cycle_ran_after

    # Map EOL back to rows
    raw_data['EOL'] = raw_data['ID'].map(
        {i+1: EOL[i][0] for i in range(len(EOL))}
    )

    # =========================
    # LIFE RATIO
    # =========================
    raw_data['LR'] = raw_data['Cycle'] / raw_data['EOL']

    # =========================
    # LABELS
    # =========================
    conditions = [
        raw_data['LR'] <= 0.6,
        (raw_data['LR'] > 0.6) & (raw_data['LR'] <= 0.8),
        raw_data['LR'] > 0.8
    ]

    raw_data['labels'] = np.select(conditions, [0, 1, 2])

    # =========================
    # CLEAN
    # =========================
    raw_data = raw_data.drop(columns=['ID', 'EOL', 'LR'])

    # =========================
    # SAVE
    # =========================
    raw_data.to_csv(f'Test_classification_{idx}.csv', index=False)

    print(f"Saved: Test_classification_{idx}.csv")

Saved: Test_classification_1.csv
Saved: Test_classification_2.csv
Saved: Test_classification_3.csv
Saved: Test_classification_4.csv
